In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [ ]:
# import the data
path = r'..\data\unzipped'
df = pd.read_csv(path + r'\Life Expectancy Data.csv')
df.head()

In [3]:
df.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness 5-9 years                 34
 thinness  1-19 years               34
 BMI                                34
Polio                               19
Diphtheria                          19
Life expectancy                     10
Adult Mortality                     10
 HIV/AIDS                            0
Country                              0
Year                                 0
Measles                              0
percentage expenditure               0
infant deaths                        0
Status                               0
under-five deaths                    0
dtype: int64

In [4]:
df.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness 5-9 years                 34
 thinness  1-19 years               34
 BMI                                34
Polio                               19
Diphtheria                          19
Life expectancy                     10
Adult Mortality                     10
 HIV/AIDS                            0
Country                              0
Year                                 0
Measles                              0
percentage expenditure               0
infant deaths                        0
Status                               0
under-five deaths                    0
dtype: int64

In [5]:
# We are droping any null values in Life Expectency column , if treated lead to data leakage
df = df[df['Life expectancy '].notnull()]

In [6]:
df.columns

Index(['Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure',
       'Diphtheria ', ' HIV/AIDS', 'GDP', 'Population',
       ' thinness  1-19 years', ' thinness 5-9 years',
       'Income composition of resources', 'Schooling'],
      dtype='object')

In [7]:
# columns to be dropped
cols = [#'Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       #'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       #'Measles ', ' BMI ', 
       'under-five deaths ', 
       #'Polio', 'Total expenditure',
       #'Diphtheria ', ' HIV/AIDS',
       'GDP','Population', ' thinness  1-19 years',
       #' thinness 5-9 years', 
       'Income composition of resources', 
       #'Schooling'
       ]
df = df.drop(columns=cols)

# these columns wer droped due to multi colinearity problem

In [ ]:
# Split BEFORE any imputation so that no statistic (median etc.) is computed from test rows.
# The data has one row per country per year, so we split by COUNTRY: each country is either
# entirely in train or entirely in test. A random row split would put e.g. India-2010 in train
# and India-2011 in test, so the model could "recognise" countries instead of generalising.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['Country']))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", train_df.shape, "countries:", train_df['Country'].nunique())
print("Test :", test_df.shape, "countries:", test_df['Country'].nunique())


def fill_with_train_median(col, by=None):
    """Fill NaNs in `col` of train_df and test_df using medians computed on train_df only."""
    for d in (train_df, test_df):
        if by is None:
            fill = train_df[col].median()
        else:
            med = train_df.groupby(by)[col].median().rename('_median')
            fill = d[by].join(med, on=by)['_median']
        d[col] = d[col].fillna(fill)

In [ ]:
# Hepatitis B
# creating a column named HepB_missing which give information to the model
for d in (train_df, test_df):
    d['Hepatitis B_missing'] = d['Hepatitis B'].isna().astype(int)

# Fill the missing values with year wise imputation (medians learned from train only)
fill_with_train_median('Hepatitis B', by=['Year'])
fill_with_train_median('Hepatitis B')

In [ ]:
# Total Expenditure

def fill_expenditure(x):
    # Forward-fill within a country only when a single year is missing.
    # Must return x unchanged otherwise: returning None (the old behaviour) turned every value
    # of those countries into NaN, which the median fallback then overwrote.
    if x.isna().sum() == 1:
        return x.ffill()
    return x

# Uses only each country's own history, and each country is wholly in train or test
for d in (train_df, test_df):
    d['Total expenditure'] = d.groupby('Country')['Total expenditure'].transform(fill_expenditure)

# global fallback for any group where even this didn't resolve everything (train median)
fill_with_train_median('Total expenditure')

print(train_df['Total expenditure'].isna().sum(), test_df['Total expenditure'].isna().sum())  # confirm 0 0

In [ ]:
# ALCOHOL
for d in (train_df, test_df):
    mask = (d['Alcohol'] == 0.01) & (d['Year'] >= 2012)
    d.loc[mask, 'Alcohol'] = np.nan
    d['Alcohol'] = d.groupby('Country')['Alcohol'].transform(lambda x: x.ffill().bfill())

fill_with_train_median('Alcohol', by=['Status', 'Year'])  # for South Sudan

In [ ]:
# SCHOOLING
fill_with_train_median('Schooling', by=['Status', 'Year'])
fill_with_train_median('Schooling', by=['Status'])

In [ ]:
# Diphtheria
for d in (train_df, test_df):
    d['Diphtheria '] = d.groupby('Country')['Diphtheria '].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

In [ ]:
train_df.shape, test_df.shape

In [19]:
df.columns

Index(['Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles ', ' BMI ', 'Polio', 'Total expenditure', 'Diphtheria ',
       ' HIV/AIDS', ' thinness 5-9 years', 'Schooling', 'Hepatitis B_missing'],
      dtype='object')

In [ ]:
# The train/test split was already made (by country) before imputation
X_train = train_df.drop(columns=['Life expectancy ', 'Country'])
X_test = test_df.drop(columns=['Life expectancy ', 'Country'])
y_train = train_df['Life expectancy ']
y_test = test_df['Life expectancy ']
country_train = train_df['Country']
country_test = test_df['Country']


label_encod = LabelEncoder()
X_train['Status'] = label_encod.fit_transform(X_train['Status'])
X_test['Status'] = label_encod.transform(X_test['Status'])

imputer = IterativeImputer(max_iter=10, random_state=0)
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns, index=X_test.index)

X_train_imputed.insert(0, 'Country', country_train.values)
X_test_imputed.insert(0, 'Country', country_test.values)

In [15]:
X_train_imputed.head()

,Country,Year,Status,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,Polio,Total expenditure,Diphtheria,HIV/AIDS,thinness 5-9 years,Schooling,Hepatitis B_missing
2268,Serbia,2012.0,1.0,126.0,1.0,9.38,742.510971,97.0,0.0,58.3,93.0,9.89,91.0,0.1,2.1,14.0,0.0
1680,Mauritius,2002.0,1.0,179.0,0.0,4.14,369.631710,88.0,0.0,26.4,88.0,4.24,88.0,0.1,7.9,12.5,0.0
2785,United Republic of Tanzania,2008.0,1.0,376.0,92.0,3.44,0.000000,86.0,3413.0,19.6,89.0,4.21,86.0,7.4,7.3,11.9,0.0
2512,Sweden,2008.0,0.0,62.0,0.0,6.90,8105.590882,92.0,25.0,56.5,98.0,9.23,98.0,0.1,1.3,15.7,1.0
1090,Guinea-Bissau,2015.0,1.0,275.0,4.0,3.57,0.000000,87.0,153.0,26.3,87.0,5.73,87.0,3.2,7.0,9.2,0.0


In [16]:
X_test_imputed.head()

,Country,Year,Status,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,Polio,Total expenditure,Diphtheria,HIV/AIDS,thinness 5-9 years,Schooling,Hepatitis B_missing
2399,South Africa,2009.0,1.0,449.0,46.0,7.60,782.598714,74.0,5857.0,46.4,75.0,8.39,76.0,19.0,9.8,12.8,0.0
196,Bangladesh,2011.0,1.0,14.0,118.0,0.01,62.349885,96.0,5625.0,15.8,96.0,3.16,96.0,0.1,19.2,9.4,0.0
2316,Singapore,2012.0,0.0,59.0,0.0,1.89,6041.858981,97.0,42.0,32.4,97.0,4.22,97.0,0.1,2.1,15.4,0.0
1735,Montenegro,2012.0,1.0,11.0,0.0,6.56,648.133178,9.0,0.0,6.2,94.0,7.25,94.0,0.1,1.9,15.1,0.0
1102,Guinea-Bissau,2003.0,1.0,38.0,5.0,2.16,2.527115,86.0,1158.0,19.0,65.0,5.62,6.0,4.6,9.5,7.4,1.0


In [ ]:
# Save
# Existing folder
out_dir = Path(r"..\data\Processed")

# Save files
X_train_imputed.to_csv(out_dir / "X_train.csv", index=False)
X_test_imputed.to_csv(out_dir / "X_test.csv", index=False)
y_train.to_csv(out_dir / "y_train.csv", index=False)
y_test.to_csv(out_dir / "y_test.csv", index=False)
